In [ ]:
import json
from pathlib import Path

OLD_BLOCK = 'if IS_GITHUB_ACTIONS:\n    assets_url = os.environ.get("DRIVE_ASSETS_URL", "").strip()\n    if not assets_url:\n        raise RuntimeError("Missing GitHub Actions secret DRIVE_ASSETS_URL (public Google Drive assets folder URL).")\n    print("Downloading Part 3 assets from Google Drive into:", ASSETS_DIR)\n    assets_download = gdown.download_folder(\n        url=assets_url,\n        output=str(ASSETS_DIR),\n        quiet=False,\n        use_cookies=False,\n    )\n    if not assets_download and not any(p.is_file() for p in ASSETS_DIR.rglob("*")):\n        raise RuntimeError("Assets download failed. Check DRIVE_ASSETS_URL and set the folder sharing to Anyone with the link — Viewer.")\n    print("Part 2 clips will be read from the downloaded GitHub artifact; no second clip download is performed.")\nelse:\n    ASSETS_FOLDER_ID = "1l9ayaXiRBwUr1Hrjag7r3CycVDiN24fh"\n    CLIPS_FOLDER_ID = "10DSjb9etdFzQTg_rftRJpJkC5jKBjsNT"\n    print("Downloading original Colab assets folder...")\n    gdown.download_folder(id=ASSETS_FOLDER_ID, output=str(ASSETS_DIR), quiet=False, use_cookies=False)\n    print("Downloading original Colab clips folder...")\n    gdown.download_folder(id=CLIPS_FOLDER_ID, output=str(INPUT_DIR), quiet=False, use_cookies=False)'

NEW_BLOCK = 'if IS_GITHUB_ACTIONS:\n    # The master workflow downloads assets/ before running this notebook.\n    # Reuse those files instead of downloading the same Google Drive folder twice.\n    existing_assets = [path for path in ASSETS_DIR.rglob("*") if path.is_file()]\n    if existing_assets:\n        print(f"Using {len(existing_assets)} asset file(s) already downloaded by the workflow.")\n    else:\n        # Keep a fallback for running this notebook directly in GitHub Actions.\n        assets_url = os.environ.get("DRIVE_ASSETS_URL", "").strip()\n        if not assets_url:\n            raise RuntimeError("Missing GitHub Actions secret DRIVE_ASSETS_URL (public Google Drive assets folder URL).")\n        print("Downloading Part 3 assets from Google Drive into:", ASSETS_DIR)\n        assets_download = gdown.download_folder(\n            url=assets_url,\n            output=str(ASSETS_DIR),\n            quiet=False,\n            use_cookies=False,\n        )\n        existing_assets = [path for path in ASSETS_DIR.rglob("*") if path.is_file()]\n        if not assets_download and not existing_assets:\n            raise RuntimeError("Assets download failed. Check DRIVE_ASSETS_URL and set the folder sharing to Anyone with the link — Viewer.")\n        if not existing_assets:\n            raise RuntimeError("The Google Drive assets folder download completed without any files.")\n    print("Part 2 clips will be read from the downloaded GitHub artifact; no second clip download is performed.")\nelse:\n    ASSETS_FOLDER_ID = "1l9ayaXiRBwUr1Hrjag7r3CycVDiN24fh"\n    CLIPS_FOLDER_ID = "10DSjb9etdFzQTg_rftRJpJkC5jKBjsNT"\n    print("Downloading original Colab assets folder...")\n    gdown.download_folder(id=ASSETS_FOLDER_ID, output=str(ASSETS_DIR), quiet=False, use_cookies=False)\n    print("Downloading original Colab clips folder...")\n    gdown.download_folder(id=CLIPS_FOLDER_ID, output=str(INPUT_DIR), quiet=False, use_cookies=False)'

import json
from pathlib import Path

original_notebook = Path("Part_3.ipynb")
if not original_notebook.is_file():
    raise FileNotFoundError("Part_3.ipynb is missing from the checked-out repository.")

notebook = json.loads(original_notebook.read_text(encoding="utf-8"))
code_cells = [cell for cell in notebook.get("cells", []) if cell.get("cell_type") == "code"]
if len(code_cells) != 1:
    raise RuntimeError(f"Expected exactly one code cell in Part_3.ipynb; found {len(code_cells)}.")

cell_source = code_cells[0].get("source", "")
if isinstance(cell_source, list):
    cell_source = "".join(cell_source)
if cell_source.count(OLD_BLOCK) != 1:
    raise RuntimeError("Could not find the expected Part 3 asset-download block exactly once; refusing to patch an unknown notebook version.")

patched_source = cell_source.replace(OLD_BLOCK, NEW_BLOCK, 1)
print("Part 3 V2: original notebook loaded; existing Drive assets will be reused.")
exec(compile(patched_source, "Part_3.ipynb:Part_3_v2", "exec"), globals(), globals())
